# Movie Rating Prediction with Matrix Completion

## Business use case

Recommendation systems create value by ranking content a user is likely to enjoy, even when each user has rated only a small fraction of the catalog. Matrix completion is a practical way to learn latent preference structure from sparse user-item feedback.

## Objective

This notebook uses MovieLens ratings to construct a sparse user-movie matrix, withhold a validation subset, and apply SoftImpute collaborative filtering. Performance is measured against a simple average-rating baseline using out-of-sample and in-sample R².

## Result in context

The stored run reports **0.1993 out-of-sample R²** and **0.6287 in-sample R²**. The model improves on the mean-rating baseline, but the large gap between train and validation performance indicates limited generalization.


## Step 1 — Prepare the collaborative-filtering environment

The notebook installs the required packages, mounts the working storage, and loads custom matrix-completion utilities together with the MovieLens ratings data.


In [179]:
# Install the standard papackages
!pip install numpy
!pip install pandas
!pip install fancyimpute

In [180]:
# location of custom packages: soft_impute , functionsCF, and dataset ratings.csv
import sys
sys.path.append('/collaborativeFiltering/')

In [181]:
# change the working directory
import os
os.chdir("/collaborativeFiltering/")

In [182]:
# Impute necessary packages
import numpy as np
import pandas as pd
from fancyimpute import BiScaler
from soft_impute import SoftImpute
from functionsCF import GenerateTrainingSet

## Step 2 — Build the sparse user-movie matrix

Only user ID, movie ID, and rating are used. Movie IDs are remapped to a compact column index, and known ratings populate an otherwise missing-value matrix.


In [183]:
# Read movielens data from files- point to where data is stored, small set of Movielens dataset
# 100836 (rows), userId	movieId	rating	timestamp (columns).
# Using smaller dataset rather than the full dataset to speed performance.
# Your results may vary depending on which Movielens data set is used; Several are available online
# read in values only
rating = pd.read_csv('ratings.csv', sep=',').values

In [184]:
# Here we only care about the ratings, so we only use the first three columns, which contain use IDs, movie IDs, and ratings.
rating = rating[:,0:3]

In [185]:
#show top 5 rows
print(rating[:5, :])

[[ 1.  1.  4.]
 [ 1.  3.  4.]
 [ 1.  6.  4.]
 [ 1. 47.  5.]
 [ 1. 50.  5.]]


In [186]:
# Use all known information to create the incomplete matrix

# First, create an empty matrix
matrix_incomplete = np.zeros((len(np.unique(rating[:,0])), len(np.unique(rating[:,1]))))

# Second, Since some movies don't have any ratings, we only use the movies that have ratings.
# Here we correspondingly change the movie IDs to make each column has ratings.
# create an array of all movie IDs
usedID = np.unique(rating[:, 1])
# replace the movie IDs by the their positions in the array we just created
for i in range(len(rating[:,1])):
    rating[:,1][i] = np.where(usedID==rating[:,1][i])[0][0] + 1

# Finally, we construct the incomplete matrix, on which the incomplete components are nan by
# default.
# all components are nan by default
matrix_incomplete[:] = np.nan
# create the index pair of the components with ratings
indices = np.array(rating[:,0] - 1).astype(int), np.array(rating[:,1] - 1).astype(int)
# change the values in the corresponding positions to the known rating information
matrix_incomplete[indices] = rating[:,2]

## Step 3 — Hold out known ratings for validation

A subset of observed user-movie pairs is withheld before model fitting. This is the correct unit of evaluation for matrix completion because the task is to predict missing ratings for known users and items.


In [187]:
# Obtain the index pairs of the training set and the validation set, with ratio 90%
train_indices, validation_indices = GenerateTrainingSet(rating[:,0], rating[:,1], 0.90)
# And then use the index pairs to create the incomplete training test
matrix_train = matrix_incomplete.copy()
matrix_train[:] = np.nan
matrix_train[train_indices] = matrix_incomplete[train_indices]

## Step 4 — Fit the low-rank matrix-completion model

The training matrix is transformed with `BiScaler` and completed with `SoftImpute`. The fitted low-rank structure estimates ratings that were not supplied to the model.


In [188]:
# Create the BiScaler model
biscaler = BiScaler(scale_rows=False, scale_columns=False, max_iters=50, verbose=False)
# Rescale both rows and columns to have zero mean
matrix_train_normalized = biscaler.fit_transform(matrix_train)

In [189]:
# Use softImpute to complete the matrix. J means the number of archetypes and rand_seed means the
# seed for the inner random number generator, verbose control whether outputting algorithm logs.
softImpute = SoftImpute(J = 4, maxit = 200, random_seed = 1, verbose = False)

In [190]:
# Run the softImpute model on the normalized training set
matrix_train_softImpute = softImpute.fit(matrix_train_normalized)
# Use the softImpute model to create the predicted matrix. If we set copyto as True, then it
# directly change the value of matrix_train_normalized
matrix_train_filled_normalized = matrix_train_softImpute.predict(matrix_train_normalized, copyto = False)
# Inverse transformation to undo the scaling
matrix_train_filled = biscaler.inverse_transform(matrix_train_filled_normalized)

## Step 5 — Compare against a baseline

The model's mean-squared error is compared with the error from predicting the training-set average rating. The resulting R² quantifies how much error the collaborative model removes relative to that baseline.


In [191]:
# Create the baseline method
train_average = np.average(matrix_train[train_indices])

In [192]:
# Calculate out-of-sample R2 and in-sample R2
# The results may vary due to datasize and training test split.
validation_mse = ((matrix_train_filled[validation_indices] - matrix_incomplete[validation_indices]) ** 2).mean()
training_mse = ((matrix_train_filled[train_indices] - matrix_incomplete[train_indices]) ** 2).mean()
validation_mse_baseline = ((train_average - matrix_incomplete[validation_indices]) ** 2).mean()
training_mse_baseline = ((train_average - matrix_incomplete[train_indices]) ** 2).mean()
print("out-of-sample R2: %.4f, in-sample R2: %.4f." % (1 - validation_mse / validation_mse_baseline, 1 - training_mse / training_mse_baseline))

out-of-sample R2: 0.1993, in-sample R2: 0.6287.


## Step 6 — Inspect the latent-factor representation

The final cells examine the learned item archetypes and user weights, then reconstruct the predicted matrix from the low-rank factors. This helps connect the recommendation output to the underlying factorization.


In [193]:
# Obtain the ratings of each archetype
# Each row of this matrix corresponds to a movie and each column corresponds to an archetype
softImpute.v

array([[-0.00110501, -0.00644537, -0.01452209, -0.00309793],
       [-0.00167285,  0.00649438, -0.0046562 ,  0.00109254],
       [ 0.00106032,  0.01188163, -0.00632373,  0.00935664],
       ...,
       [ 0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ]])

In [194]:
softImpute.v.shape

(9724, 4)

In [195]:
# Obtain the weights of archetypes of each user
# each row of this matrix corresponds to a user and each column corresponds to an archetype
weights = np.dot(softImpute.u, np.diagflat(softImpute.d).T)
weights

array([[ -0.91904527, -10.55162483, -18.62152621,  -0.32241214],
       [-11.84476422,   8.03294299,  -6.31192253,  -0.55644328],
       [-45.07610782,   8.19300557,  82.99620653,  -9.81614088],
       ...,
       [-23.34433363,  10.15403815,   6.66359852, -27.67989031],
       [ -2.99533417,   2.49057309,  -0.67929737,   2.17296334],
       [  0.25288116,   5.69102876,   2.42982131,  24.4901997 ]])

In [196]:
weights.shape

(610, 4)

In [197]:
# And then the predicted matrix is computed by the product of two low-rank matrices
new_prediction = np.dot(weights, softImpute.v.T)

In [198]:
# We can see it is the same with the output of the codes in the previous section
np.sum(np.abs(new_prediction - matrix_train_filled_normalized))

np.float64(7.476050950806106e-11)

## Technical conclusions

The model produces **0.1993 out-of-sample R²** versus **0.6287 in-sample R²**. It is learning meaningful latent structure, but the generalization gap could be perceived as substantial. However, the near-zero reconstruction difference shown in the notebook confirms that the stored low-rank factors reproduce the completed normalized matrix as expected.

## Business conclusions

The current model is better suited as a candidate-generation or personalization baseline than as a final production recommender. Its value is in proving that sparse ratings contain reusable latent preference signals; the next business question is whether those signals improve engagement when used to rank actual recommendations.

## Limitations and next steps

The evaluation predicts withheld ratings rather than measuring top-N recommendation quality. Cold-start users/items and temporal changes in taste are also outside the current design.
